[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/b_14_cross_entropy_fused.ipynb)

# 🟡 Medium: Cross-Entropy Without logsumexp

*Training*
The same loss as **problem 16**, with the library function taken away:

$$\ell_i = \operatorname{logsumexp}(z_i) - z_{i,t_i}, \qquad
\operatorname{logsumexp}(z) = \log \sum_k e^{z_k}$$

Write that `logsumexp` yourself. Return the **mean over the batch**.

### Rules
- Signature: `cross_entropy_loss(logits, targets)` — identical to problem 16
- `logits` is `(B, C)`, `targets` is `(B,)` of integer class ids; the output is a **scalar**
- Banned: **`jax.scipy.special.logsumexp`** and `jax.nn.logsumexp`, plus
  `jax.nn.log_softmax`, `jax.nn.softmax`, `optax`
- Only `jnp.max`, `jnp.exp`, `jnp.log`, `jnp.sum` and friends
- Must stay finite at extreme logits — including in `float16` — and work under `jit`

### The shift, and why the naive route dies twice

$$\log \sum_k e^{z_k} \;=\; m + \log \sum_k e^{z_k - m}, \qquad m = \max_k z_k$$

The identity is exact: pull $e^m$ out of the sum and the $\log$ turns it into a
`+m`. What it buys is that every exponent is now $\le 0$, so the largest term is
exactly `1.0` and the sum lands in $[1, C]$ — it cannot overflow, whatever the
inputs were.

**Overflow.** Without the shift, `exp(z)` is `inf` above $z \approx 88.7$ in
float32, and above $z \approx 11.1$ in **float16** — which is why this bites in
mixed-precision training long before anyone's logits look extreme.

**Underflow — the one that actually bites.** Even with no overflow, a
confidently *wrong* prediction pushes $p_t$ under the float32 floor: normals
stop at $\approx 1.2\times10^{-38}$, subnormals at $\approx 1.4\times10^{-45}$,
and XLA flushes subnormals to zero on accelerators anyway. Once $p_t$ rounds to
`0.0`, `log(0) = -inf` makes the loss `inf` and every gradient `nan`. The fused
form never materialises $p_t$: it computes $z_t - \log\sum_k e^{z_k}$, a
perfectly finite number like $-120$ (that is $p_t \approx 10^{-52}$, hopelessly
unrepresentable, yet its logarithm is an ordinary float). Your loss stays
large-but-finite and training recovers instead of poisoning every parameter
with `nan`.

There is a gradient bonus too. $\partial \ell / \partial z = p - q$ — a clean,
bounded expression that autodiff derives exactly from the fused form. Compose
`log` on top of a separate `softmax` and you hand XLA a division of two tiny
numbers to differentiate through.

### stop_gradient on the max
$\ell$ is invariant to the shift — it cancels analytically — so $m$ carries no
gradient information. `jax.lax.stop_gradient(m)` says so explicitly: the answer
and its derivatives are unchanged, and the backward pass no longer threads
through the `max`'s argmax-shaped subgradient. Leaving it out is not *wrong*
here; knowing why it costs nothing is the point.

### Watch the axis
`jnp.max(logits, axis=-1)` drops the class axis, so `logits - m` fails to
broadcast (or worse, broadcasts against the batch axis when `B == C` and
silently gives a wrong answer). `keepdims=True` on both the `max` and the `sum`
is what keeps the shapes honest — the same discipline as **`b_12`**.

**`b_15`** takes this further with label smoothing and a padding mask.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp


def cross_entropy_loss(logits, targets):
    """Mean cross-entropy over the batch — logsumexp written by hand.

    Args:
        logits:  (B, C) unnormalised scores
        targets: (B,) integer class ids

    Returns:
        Scalar loss.
    """
    pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp

# Uniform logits over 3 classes -> loss is exactly log(3).
print("uniform:", cross_entropy_loss(jnp.zeros((1, 3)), jnp.array([0])), "vs", jnp.log(3.0))

# Against the library routines you are not allowed to call.
logits = jax.random.normal(jax.random.key(0), (4, 5)) * 3.0
targets = jnp.array([1, 2, 0, 4])
ref = -jnp.mean(jnp.take_along_axis(
    jax.nn.log_softmax(logits, axis=-1), targets[:, None], axis=-1))
print("mine:", float(cross_entropy_loss(logits, targets)), " ref:", float(ref))

# What the shift is worth. float32 first: exp(1000) = inf.
big = jnp.array([[1000.0, 0.0, 0.0]])
print("\nshifted   :", float(cross_entropy_loss(big, jnp.array([0]))))
print("unshifted :", float(jnp.log(jnp.sum(jnp.exp(big))) - big[0, 0]), " <- inf - inf")

# float16 overflows at exp(11.1), so this is not an exotic case at all.
h = jnp.array([[20.0, 0.0, 0.0]], dtype=jnp.float16)
print("\nfp16 shifted  :", float(cross_entropy_loss(h, jnp.array([0]))))
print("fp16 exp(20)  :", float(jnp.exp(h[0, 0])), " <- inf, in a dtype people train in")

# Confidently WRONG stays finite: ~1000, not inf.
print("\nwrong class:", float(cross_entropy_loss(big, jnp.array([1]))))

# The gradient is p - onehot, averaged over the batch.
g = jax.grad(cross_entropy_loss)(jnp.array([[2.0, 1.0, 0.0]]), jnp.array([0]))
print("grad:", g, " sums to", float(jnp.sum(g)))

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution, status

check("cross_entropy_fused")

# hint("cross_entropy_fused")      # stuck? nudge without the answer
# solution("cross_entropy_fused")  # spoiler: the reference implementation
# status()                         # your dashboard across all problems